# `mini-conductor`: Colab H100 / A100 Quickstart

Reproducing & extending Sakana AI's **The Conductor** ([arXiv:2512.04388](https://arxiv.org/abs/2512.04388)) on a single Colab H100 80GB + Claude API.

**Pipeline:**
1. Mount Google Drive (for crash-proof checkpoints & SQLite cache)
2. Install dependencies & set `ANTHROPIC_API_KEY`
3. Run unit tests (`pytest -v`)
4. Build Math / Code / GPQA datasets
5. **Stage 0:** Synthesize verified trajectories & run 10-min LoRA SFT warm-start
6. **Stage 1:** Run GRPO experiments (**RQ1–RQ4**)
7. Run evaluation suite & export figures

In [ ]:
# 1. Mount Google Drive & check GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# 2. Point to your uploaded/cloned mini-conductor directory & install requirements
import os
PROJECT_DIR = '/content/drive/MyDrive/mini-conductor'  # Edit if placed elsewhere
os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt
!pytest -v

In [ ]:
# 3. Configure Anthropic API Key (and optional HuggingFace / W&B tokens)
import os
from getpass import getpass

os.environ['ANTHROPIC_API_KEY'] = getpass('Enter ANTHROPIC_API_KEY: ')
# Optional: needed only if downloading gated Idavidrein/gpqa
# os.environ['HF_TOKEN'] = getpass('Enter HF_TOKEN (optional): ')

In [ ]:
# 4. Build unified Math (MATH Level 3-5 / MATH-500) + Code (MBPP) + GPQA/MMLU-Pro datasets
!python scripts/01_build_dataset.py --output_dir data

In [ ]:
# 5. Stage 0: Synthesize & verify 300 diverse-topology warm-start trajectories via Claude Sonnet
!python scripts/02_gen_sft_data.py \
  --pool configs/pool_claude_hybrid.yaml \
  --dataset data/train.jsonl \
  --output data/sft_warmup.jsonl \
  --num_samples 300 \
  --allow_recursion \
  --permute_ids

# Run 2-epoch LoRA SFT warm-start (~10 min on H100)
!python scripts/03_train_sft.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --data data/sft_warmup.jsonl \
  --output_dir checkpoints/sft_warmup \
  --epochs 2

In [ ]:
# 6A. Stage 1 Main Run (RQ1 + RQ2 All-Depth Recursive + RQ3 ID-Permutation)
!python scripts/04_train_grpo.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --init_adapter checkpoints/sft_warmup \
  --pool configs/pool_claude_hybrid.yaml \
  --train_data data/train.jsonl \
  --credit_mode all_depth \
  --allow_recursion \
  --permute_ids \
  --cost_lambda 0.05 \
  --batch_size 6 \
  --group_size 8 \
  --total_steps 160 \
  --save_dir checkpoints/grpo_all_depth_perm \
  --resume

In [ ]:
# 6B. RQ2 Controlled Ablation: Top-Level Only GRPO (same seed & pool)
!python scripts/04_train_grpo.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --init_adapter checkpoints/sft_warmup \
  --pool configs/pool_claude_hybrid.yaml \
  --train_data data/train.jsonl \
  --credit_mode top_level \
  --allow_recursion \
  --permute_ids \
  --cost_lambda 0.05 \
  --batch_size 6 \
  --group_size 8 \
  --total_steps 160 \
  --save_dir checkpoints/grpo_top_level_perm \
  --resume

In [ ]:
# 7. Run Full Evaluation Suite (Single Models, Recursion Ablation, ID Swap, Zero-Shot Claude Opus)
!python scripts/05_eval_suite.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --adapter checkpoints/grpo_all_depth_perm/latest \
  --pool configs/pool_claude_hybrid.yaml \
  --eval_data data/eval_all.jsonl \
  --include_held_out \
  --cost_aware \
  --output_dir results/rq_eval